In [ ]:
import os
import glob
import json
import numpy as np
import pandas as pd
# imported libraries

In [ ]:
# Build master dataframe by loading all analyzed JSON result files
# and extracting model metadata, prompt framing, demographics, and
# response information for each simulated respondent iteration.

base_path = "../prompting results/02_final_results/"

# Recursively collect all analyzed JSON output files
all_files = glob.glob(
    os.path.join(base_path, "**", "analyzed_*.json"),
    recursive=True
)

rows = []

for file in all_files:

    # Extract survey, question, and model provider from folder structure
    rel_path = os.path.relpath(file, base_path)
    parts = rel_path.split(os.sep)

    survey = parts[0].strip()
    question = parts[1].strip()
    model_provider = parts[2].strip()

    # Standardize model provider names
    if model_provider == "deepseek-ai":
        model = "DeepSeek"
    elif model_provider == "gpt-5-nano":
        model = "GPT-5 Nano"
    elif model_provider == "meta-llama":
        model = "Meta-Llama"
    else:
        model = model_provider

    filename = os.path.basename(file).lower()

    # Determine prompt framing based on filename
    # (embody, specialist, or direct prompting)
    if "embody" in filename:
        prompt_frame = "embody"
    elif "specialist" in filename:
        prompt_frame = "specialist"
    else:
        prompt_frame = "direct"

    # Create combined survey-question identifier
    survey_question = f"{survey}_{question}"

    # Load JSON results file
    with open(file, "r") as f:
        data = json.load(f)

    # Iterate through each simulated respondent entry
    for entry in data:

        # Extract demographic attributes
        demo = entry.get("demographics", {})

        # Extract the model's selected response option
        extracted = entry.get("extracted_option")

        # Flag refusals (used for refusal-rate analysis)
        refusal = 1 if extracted == "refusal" else 0

        # GPT models include reasoning effort metadata
        if "gpt" in model.lower():
            reasoning_effort = entry.get("reason_effort", "unknown")
        else:
            reasoning_effort = "N/A"

        # Append row for dataframe construction
        rows.append({
            "model": model,
            "prompt_frame": prompt_frame,
            "survey_question": survey_question,
            "poll_name": entry.get("poll_name"),
            "iteration": entry.get("iteration"),
            "reasoning_effort": reasoning_effort,
            "extracted_option": extracted,
            "race": demo.get("race_or_ethnicity"),
            "gender": demo.get("gender"),
            "age": demo.get("age"),
            "education": demo.get("education"),
            "income": demo.get("income"),
            "party": demo.get("party"),
            "refusal": refusal
        })

# Convert collected rows into a pandas dataframe
whole_df = pd.DataFrame(rows)



# Define response scale mappings to convert textual Likert responses
# into numeric ordinal values for quantitative analysis.

frequency_5pt_map = {"all the time": 5, "often": 4, "sometimes": 3, "rarely": 2, "never": 1}
support_oppose_5pt_map = {"strongly support": 5, "somewhat support": 4, "don’t know": 3, "somewhat oppose": 2, "strongly oppose": 1}
sd_7pt_map = {"1": 1, "2": 2, "3": 3, "4": 4, "5": 5, "6": 6, "7": 7}
support_oppose_ns_5pt_map = {"strongly approve": 5, "somewhat approve": 4, "not sure": 3, "somewhat disapprove": 2, "strongly disapprove": 1}
support_oppose_4pt_map = {"strongly support": 4, "somewhat support": 3, "somewhat oppose": 2, "strongly oppose": 1}
support_neutral_oppose_5pt_map = {"strongly support": 5, "somewhat support": 4, "neither support nor oppose": 3, "somewhat oppose": 2, "strongly oppose": 1}
priority_6pt_map = {"highest priority": 6, "a high priority but not the highest priority": 5, "somewhat of a priority": 4, "low priority": 3, "not a priority at all": 2, "no answer": 1}
problem_5pt_map = {"a very serious problem": 5, "a somewhat serious": 4, "not sure": 3, "a minor problem": 2, "not a problem": 1}
available_5pt_map = {"very available": 5, "somewhat available": 4, "don't know": 3, "not very available": 2, "not at all available": 1}
confident_6pt_map = {"very confident": 6, "somewhat confident": 5, "don't know": 4, "not very confident": 3, "not at all confident": 2, "prefer not to say": 1}


# Map each survey question to its corresponding Likert response scale
likert_outcome_maps = {
    "AFSP_Q4": frequency_5pt_map,
    "progress_Q1": support_oppose_5pt_map,
    "progress_Q2": support_oppose_5pt_map,
    "progress_Q3": support_oppose_5pt_map,
    "progress_Q4": support_oppose_5pt_map,
    "progress_Q5": support_oppose_5pt_map,
    "progress_Q6": support_oppose_5pt_map,
    "ipsos_Q1": priority_6pt_map,
    "ipsos_Q2": priority_6pt_map,
    "ipsos_Q3": priority_6pt_map,
    "ipsos_Q4": priority_6pt_map,
    "ipsos_Q5": priority_6pt_map,
    "ipsos_Q6": priority_6pt_map,
    "ipsos_Q7": priority_6pt_map,
    "ipsos_Q8": support_oppose_4pt_map,
    "ipsos_Q9": support_oppose_4pt_map,
    "ipsos_Q10": support_oppose_4pt_map,
    "science_direct_Q1": sd_7pt_map,
    "science_direct_Q2": sd_7pt_map,
    "science_direct_Q3": sd_7pt_map,
    "science_direct_Q4": sd_7pt_map,
    "science_direct_Q5": sd_7pt_map,
    "duke_Q1": support_neutral_oppose_5pt_map,
    "YouGOV_Biden_Trump_Q1": problem_5pt_map,
    "YouGOV_Biden_Trump_Q2": support_oppose_ns_5pt_map,
    "YouGOV_Biden_Trump_Q3": support_oppose_ns_5pt_map,
    "YouGOV_Suicide_prevention_Q1": available_5pt_map,
    "YouGOV_Suicide_prevention_Q3": confident_6pt_map
}

# Normalize dictionary keys to lowercase to avoid case mismatches
likert_outcome_maps = {k.lower(): v for k, v in likert_outcome_maps.items()}


# Identify response type (binary vs ordinal) for each survey question

def assign_response_type(q):
    base = q.split('_')[-2] + '_' + q.split('_')[-1]
    if base in ["AFSP_Q1", "AFSP_Q2", "AFSP_Q3", "duke_Q1"]:
        return "binary"
    return "ordinal"

whole_df["response_type"] = whole_df["survey_question"].apply(assign_response_type)



# Convert textual response options into numeric values
# using the appropriate response scale mapping

def map_response(row):

    val = row["extracted_option"]

    if val is None:
        return np.nan

    val_clean = str(val).strip().lower()

    # Binary conversion
    if row["response_type"] == "binary":
        if val_clean == "true":
            return 1
        if val_clean == "false":
            return 0
        return np.nan

    # Ordinal Likert conversion
    mapping = likert_outcome_maps.get(row["survey_question"].lower(), {})
    mapping_lower = {k.lower(): v for k, v in mapping.items()}

    return mapping_lower.get(val_clean, np.nan)


# Create analysis dataset restricted to ordinal response questions
# (used for distributional accuracy and error analysis)

analysis_df = whole_df[
    (whole_df["response_type"] == "ordinal")
].copy()


# Diagnostic checks to confirm survey-question coverage

print("All survey questions in dataset:")
print(sorted(whole_df["survey_question"].unique()))

print("\nSurvey questions used in analysis:")
print(sorted(analysis_df["survey_question"].unique()))

All survey questions in dataset:
['AFSP_Q1', 'AFSP_Q2', 'AFSP_Q3', 'AFSP_Q4', 'YouGOV_Biden_Trump_Q1', 'YouGOV_Biden_Trump_Q2', 'YouGOV_Biden_Trump_Q3', 'YouGOV_Suicide_prevention_Q1', 'YouGOV_Suicide_prevention_Q2', 'YouGOV_Suicide_prevention_Q3', 'duke_Q1', 'ipsos_Q1', 'ipsos_Q10', 'ipsos_Q2', 'ipsos_Q3', 'ipsos_Q4', 'ipsos_Q5', 'ipsos_Q6', 'ipsos_Q7', 'ipsos_Q8', 'ipsos_Q9', 'progress_Q1', 'progress_Q2', 'progress_Q3', 'progress_Q4', 'progress_Q5', 'progress_Q6', 'science_direct_Q1', 'science_direct_Q2', 'science_direct_Q3', 'science_direct_Q4', 'science_direct_Q5']

Survey questions used in analysis:
['AFSP_Q4', 'YouGOV_Biden_Trump_Q1', 'YouGOV_Biden_Trump_Q2', 'YouGOV_Biden_Trump_Q3', 'YouGOV_Suicide_prevention_Q1', 'YouGOV_Suicide_prevention_Q2', 'YouGOV_Suicide_prevention_Q3', 'ipsos_Q1', 'ipsos_Q10', 'ipsos_Q2', 'ipsos_Q3', 'ipsos_Q4', 'ipsos_Q5', 'ipsos_Q6', 'ipsos_Q7', 'ipsos_Q8', 'ipsos_Q9', 'progress_Q1', 'progress_Q2', 'progress_Q3', 'progress_Q4', 'progress_Q5', 'progress

In [ ]:
len(analysis_df) # removed the 4 surveys that did not have ground truth values

772635

In [ ]:
# Export full combined CSV
whole_df.to_csv("LLM_combined_results.csv", index=False)

# Export analysis subset
analysis_df.to_csv("LLM_analysis_ordinal.csv", index=False)

In [ ]:
import pandas as pd

file_path = "S2 Ground Truth Data.xlsx"

# Read all sheets
all_sheets = pd.read_excel(file_path, sheet_name=None)

tidy_dfs = []

for sheet_name, df in all_sheets.items():

    # Convert all column names to strings and strip whitespace
    df.columns = df.columns.astype(str).str.strip()

    # ID columns to keep
    id_cols = ["Survey No.", "Question_ID", "Poll name", "Group", "Subgroup"]

    print(f"Sheet: {sheet_name}") # provide updates to progress for quality assurance
    print(f"Columns read: {df.columns.tolist()}")

    # Skip sheets missing required ID columns
    if not all(col in df.columns for col in id_cols):
        print(f"Skipping sheet '{sheet_name}' !!!! missing required ID columns") #<- fail safe
        continue

    # Convert all column names to strings
    cols_str = [str(c) for c in df.columns]

    # Detect AOR columns (ends with "_aor", case-insensitive)
    aor_cols = [c for c in cols_str if str(c).lower().endswith("_aor")] # these are removed because our ground truth comes from population response percentages but we wanted to be sure to account for these responses

    # Percent columns: all other columns except ID and AORs
    percent_cols = [c for c in cols_str if c not in id_cols + aor_cols]

    # Melt percent columns
    percent_long = df.melt(
        id_vars=id_cols,
        value_vars=percent_cols,
        var_name="response_option",
        value_name="percent"
    )
    percent_long["response_option"] = percent_long["response_option"].astype(str)

    # Melt AOR columns (if they exist)
    if aor_cols:
        aor_long = df.melt(
            id_vars=id_cols,
            value_vars=aor_cols,
            var_name="response_option",
            value_name="AOR"
        )
        # Strip the "_aor" suffix
        aor_long["response_option"] = aor_long["response_option"].str.replace("_aor$", "", regex=True)
        aor_long["response_option"] = aor_long["response_option"].astype(str)

        # Merge percent + AOR
        df_long = percent_long.merge(
            aor_long,
            on=id_cols + ["response_option"],
            how="left"
        )
    else:
        df_long = percent_long
        df_long["AOR"] = pd.NA

    # Rename columns for ease of use
    df_long = df_long.rename(columns={
        "Survey No.": "survey_no",
        "Question_ID": "question_id",
        "Poll name": "poll_name",
        "Group": "subgroup_type",
        "Subgroup": "subgroup"
    })

    tidy_dfs.append(df_long)

# Combine all sheets
ground_truth_tidy = pd.concat(tidy_dfs, ignore_index=True)

# Save to CSV
ground_truth_tidy.to_csv("ground_truth_tidy.csv", index=False)

print("Done.")
print(ground_truth_tidy.head())

Sheet: duke
Columns read: ['Survey No.', 'Question_ID', 'Poll name', 'Group', 'Subgroup', 'strongly support_aor']
Sheet: AFSP
Columns read: ['Survey No.', 'Question_ID', 'Poll name', 'Group', 'Subgroup', 'often']
Sheet: science_direct
Columns read: ['Survey No.', 'Question_ID', 'Poll name', 'Group', 'Subgroup', '7', '7_aor', '6', '6_aor']
Sheet: ipsos
Columns read: ['Survey No.', 'Question_ID', 'Poll name', 'Group', 'Subgroup', 'High priority', 'A high priority, but not\nthe highest priority', 'Somewhat of a priority', 'Low priority', 'Not a priority at all']
Sheet: ipsos8-10
Columns read: ['Survey No.', 'Question_ID', 'Poll name', 'Group', 'Subgroup', 'strongly support', 'somewhat support', 'somewhat oppose', 'strongly oppose']
Sheet: progress
Columns read: ['Survey No.', 'Question_ID', 'Poll name', 'Group', 'Subgroup', 'Strongly support', 'Somewhat support', 'Somehwat Oppose', 'Strongly oppose', "Don't Know"]
Sheet: YouGOV_Biden_Trump1
Columns read: ['Survey No.', 'Question_ID', 'Pol

In [56]:
print(sorted(ground_truth_tidy['poll_name'].unique()))

['AFSP_Mental_Health_Q4', 'Biden_and_Trump_Handling_of_Problems_Q1', 'Biden_and_Trump_Handling_of_Problems_Q2', 'Biden_and_Trump_Handling_of_Problems_Q3', 'DATA_FOR_PROGRESS_Q1', 'DATA_FOR_PROGRESS_Q2', 'DATA_FOR_PROGRESS_Q3', 'DATA_FOR_PROGRESS_Q4', 'DATA_FOR_PROGRESS_Q5', 'DATA_FOR_PROGRESS_Q6', 'IPSOS_KP_988_awareness_Q1', 'IPSOS_KP_988_awareness_Q10', 'IPSOS_KP_988_awareness_Q2', 'IPSOS_KP_988_awareness_Q3', 'IPSOS_KP_988_awareness_Q4', 'IPSOS_KP_988_awareness_Q5', 'IPSOS_KP_988_awareness_Q6', 'IPSOS_KP_988_awareness_Q7', 'IPSOS_KP_988_awareness_Q8', 'IPSOS_KP_988_awareness_Q9', 'Science_Direct_suicide_help_Q1', 'Science_Direct_suicide_help_Q2', 'Science_Direct_suicide_help_Q3', 'Science_Direct_suicide_help_Q4', 'Science_Direct_suicide_help_Q5', 'Suicide_prevention_Q1', 'Suicide_prevention_Q2', 'Suicide_prevention_Q3']
